In [1]:
import pandas as pd

c:\users\никита\appdata\local\programs\python\python39\lib\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [38]:
costs_df1 = pd.read_excel("data/Распределенные счета на оплату 3800-2023.XLSX")
# costs_df2 = pd.read_excel("data/Распределенные счета на оплату 4200-4000-3800-2024.XLSX")
# costs_df3 = pd.read_excel("data/Распределенные счета на оплату 5400-2023.XLSX")
# costs_df4 = pd.read_excel("data/Распределенные счета на оплату 5400-2024.XLSX")
# costs_df5 = pd.read_excel("data/Распределенные счета на оплату 5500-2023.XLSX")
# costs_df = pd.concat([costs_df1, costs_df2, costs_df3, costs_df4, costs_df5], axis=0)

In [39]:
costs_df1['Признак "Использование в основной деятельности"'] = ~costs_df1['Признак "Использование в основной деятельности"'].isna()
costs_df1['Признак "Способ использования"'] = ~costs_df1['Признак "Способ использования"'].isna()
costs_df1 = costs_df1.dropna(subset=["Счет главной книги"])


In [43]:
costs_df1["Счет главной книги"] = costs_df1["Счет главной книги"].astype("int64")

In [44]:
costs_df1["Счет главной книги"].value_counts()

Счет главной книги
7048209010    228071
7048208010     70736
7047505010     36078
7048414960     13931
7048406010     13608
7047504010      8302
7048414970      5305
7048209020      4142
7048208020      1185
7047805010       736
7047505020       658
7047504020       170
7048406020       119
7048401020         6
Name: count, dtype: int64

In [48]:
costs_df1

,Компания,Год счета,Номер счета,Позиция счета,Номер позиции распределения,Дата отражения в учетной системе,ID договора,Услуга,Класс услуги,Здание,Класс ОС,ID основного средства,"Признак ""Использование в основной деятельности""","Признак ""Способ использования""",Площадь,Сумма распределения,Счет главной книги
0,3800,2023,5006159023,1,1,2023-01-09,ДПН 3800/74368,800003254,S004,ЗДН 3800/1/511,91507998,38009150000164290,True,False,55.0,52.06,7048209010
1,3800,2023,5006159023,1,2,2023-01-09,ДПН 3800/74368,800003254,S004,ЗДН 3800/1/511,60401018,38006040007124050,True,False,590.7,559.10,7048209010
2,3800,2023,5006159023,2,1,2023-01-09,ДПН 3800/74368,800003261,S004,ЗДН 3800/1/511,91507998,38009150000164290,True,False,55.0,68.09,7048209010
3,3800,2023,5006159023,2,2,2023-01-09,ДПН 3800/74368,800003261,S004,ЗДН 3800/1/511,60401018,38006040007124050,True,False,590.7,731.31,7048209010
4,3800,2023,5006159026,1,1,2023-01-09,ДПН 3800/74367,800001855,S004,ЗДН 3800/1/511,91507998,38009150000164290,True,False,55.0,1370.87,7048209010
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
423564,3800,2023,5006666174,1,1,2023-12-28,ДПН 3800/83501,800007790,S004,ЗДН 3800/1/2599,60401018,38006040051769750,True,False,664.2,642.99,7048209010
423565,3800,2023,5006666174,1,2,2023-12-28,ДПН 3800/83501,800007790,S004,ЗДН 3800/1/2599,60401018,38006040051769752,True,True,1.0,0.97,7048209010
423566,3800,2023,5006666174,1,3,2023-12-28,ДПН 3800/83501,800007790,S004,ЗДН 3800/1/2599,60401018,38006040051769751,True,True,1.0,0.97,7048209010
423567,3800,2023,5006666398,1,1,2023-12-29,ДПН 3800/83504,800007790,S004,ЗДН 3800/1/164,60401018,38006040010141330,True,False,514.4,4206.80,7048209010


In [54]:
cat_features = ["Компания", "Год счета", "Позиция счета", "Номер позиции распределения",
                "Услуга", "Класс услуги", "Класс ОС", 'Признак "Использование в основной деятельности"', 'Признак "Способ использования"']
x_features = ["Площадь", "Сумма распределения"]

In [59]:
costs_df1["Класс ОС"].fillna(0, inplace=True)

C:\Users\843E~1\AppData\Local\Temp/ipykernel_8392/457616636.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  costs_df1["Класс ОС"].fillna(0, inplace=True)


In [61]:
X = costs_df1[x_features + cat_features]
y = costs_df1["Счет главной книги"].astype(str)

In [73]:
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

train_pool = Pool(X_train, y_train, cat_features=cat_features)
test_pool = Pool(X_test, y_test, cat_features=cat_features)

In [81]:
clf = CatBoostClassifier(eval_metric="TotalF1", early_stopping_rounds=10)
clf.fit(train_pool, eval_set=test_pool)

Learning rate set to 0.121075
0:	learn: 0.9898445	test: 0.9899568	best: 0.9899568 (0)	total: 5.34s	remaining: 1h 28m 54s
1:	learn: 0.9947428	test: 0.9948621	best: 0.9948621 (1)	total: 9.64s	remaining: 1h 20m 9s
2:	learn: 0.9951605	test: 0.9952930	best: 0.9952930 (2)	total: 14.1s	remaining: 1h 18m 2s
3:	learn: 0.9979765	test: 0.9981270	best: 0.9981270 (3)	total: 18.4s	remaining: 1h 16m 18s
4:	learn: 0.9980029	test: 0.9981402	best: 0.9981402 (4)	total: 22s	remaining: 1h 12m 58s
5:	learn: 0.9979963	test: 0.9981402	best: 0.9981402 (4)	total: 26.3s	remaining: 1h 12m 31s
6:	learn: 0.9993096	test: 0.9995237	best: 0.9995237 (6)	total: 30.8s	remaining: 1h 12m 54s
7:	learn: 0.9993652	test: 0.9995629	best: 0.9995629 (7)	total: 36.1s	remaining: 1h 14m 37s
8:	learn: 0.9993717	test: 0.9995237	best: 0.9995629 (7)	total: 40s	remaining: 1h 13m 21s
9:	learn: 0.9994109	test: 0.9995760	best: 0.9995760 (9)	total: 43.5s	remaining: 1h 11m 42s
10:	learn: 0.9994501	test: 0.9996282	best: 0.9996282 (10)	total: 4

In [82]:
preds = clf.predict(X_test)

from sklearn.metrics import accuracy_score, f1_score
print("acc: ", accuracy_score(y_test, preds))
print("f1: ", f1_score(y_test, preds, average="macro"))

acc:  0.9998564156115389
f1:  0.9261682042823908


In [83]:
clf.get_feature_importance(prettified=True)

,Feature Id,Importances
0,Класс услуги,57.808090
1,"Признак ""Использование в основной деятельности""",15.639920
2,Класс ОС,15.072738
3,Услуга,7.195910
4,"Признак ""Способ использования""",4.283341
5,Площадь,0.000000
6,Сумма распределения,0.000000
7,Компания,0.000000
8,Год счета,0.000000
9,Позиция счета,0.000000
